In [2]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget
from matplotlib.colors import Normalize

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

from analytics_processing.modality_loading import session_modality_from_nas


import scipy.stats as stats

In [4]:
# first set the paths and logger. You can see all kind of outputs in this directory 
# already. Poster used plots from this directory.

data = {}
nas_dir = C.device_paths()[0]
output_dir = f"../outputs/validation/vel_corr/"
Logger().init_logger(None, None, logging_level="WARNING")

In [5]:
# analysis was intially done on all sessions from animal 6 with paradigm 1100
# But the poster focused on S8-15

animal_ids = [10]
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [6]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [8]:
kinematics = analytics.get_analytics('BehaviorFramewise', session_names=session_names)#[:3])



/home/vrmaster/Projects/VirtualReality/analysisVR/analytics_processing/analytics.py:593: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  aggr = pd.concat(aggr)


In [9]:
kinematics.columns

Index(['frame_id', 'frame_pc_timestamp', 'frame_position',
       'frame_ephys_timestamp', 'trial_id', 'frame_ephys_patched', 'fps',
       'track_zone', 'track_zone_int', 'from_cm_position_bin',
       'to_cm_position_bin', 'frame_velocity', 'frame_acc', 'frame_raw',
       'frame_yaw', 'frame_pitch', 'frame_raw_500msMedian',
       'frame_yaw_500msMedian', 'frame_pitch_500msMedian',
       'frame_raw_abs_acc_500msMedian', 'frame_yaw_abs_acc_500msMedian',
       'frame_pitch_abs_acc_500msMedian', 'frame_RawYawPitch_abs_vel_sum',
       'frame_RawYawPitch_abs_vel_sum_500msMedian',
       'frame_RawYawPitch_abs_acc_sum_500msMedian',
       'frame_YawPitch_abs_vel_sum_500msMedian',
       'frame_YawPitch_abs_acc_sum_500msMedian', 'forward_vs_rotation_corr',
       'frame_forward_prop', 'frame_below_velocity_thr', 'lick_count',
       'lick_mean_value', 'reward-removed_count', 'reward-removed_mean_value',
       'reward-sound_count', 'reward-sound_mean_value',
       'reward-valve-open_co

In [ ]:
plt.close('all')
plt.figure()
for track_zone in kinematics['track_zone'].unique():
    plt.title(f"Track zone: {track_zone}")
    plt.hist(kinematics[kinematics['track_zone'] == track_zone]['from_cm_position_bin'], bins=30, label=track_zone, alpha=0.5)
plt.legend()    
plt.show()
    # print(kinematics[kinematics['track_zone'] == track_zone]['from_cm_position_bin'].isna().sum(), kinematics[kinematics['track_zone'] == track_zone].shape[0])

In [ ]:
plt.close('all')

fig, ax = plt.subplots(figsize=(10, 6))

for i, s_id in enumerate(kinematics.index.unique('session_id')):
    # vals = kinematics.xs(s_id, level='session_id')['fps'].dropna()
    # percentile_95 = np.percentile(vals, 95)
    vals = kinematics.xs(s_id, level='session_id')['fps'].dropna()
    percentile_95 = np.percentile(vals, .95)
    print(i, s_id)
    
    # Only label sessions with low FPS (< 50)
    lbl = f'Session {i}: {s_id} (95%: {percentile_95:.1f})' if percentile_95 < 50 else None
    
    # Use different colors and line styles
    alpha = 1.0 if percentile_95 < 50 else 0.3
    linewidth = 2 if percentile_95 < 50 else 1
    
    ax.hist(vals, cumulative=True, density=True, histtype='step',
            label=lbl, alpha=alpha, linewidth=linewidth)

# Add labels and title
ax.set_xlabel('FPS (Frames Per Second)', fontsize=12)
ax.set_ylabel('Cumulative Density', fontsize=12)
ax.set_title('Cumulative Distribution of FPS Across Sessions\n(Low FPS sessions highlighted)', 
             fontsize=13, pad=15)

# Set y-limits and add grid
ax.set_ylim(0, .35)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Clean up spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend outside plot area
if ax.get_legend_handles_labels()[0]:  # Only if there are labeled items
    ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9, framealpha=0.95)

plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')
plt.plot(kinematics.frame_position[:7000].reset_index(drop=True))
plt.show()

plt.figure()
plt.hist(kinematics.frame_position.values, bins=1000)
plt.show()

for s_id in kinematics.index.get_level_values('session_id').unique():
    n_na = np.isnan(kinematics['frame_raw'].xs(s_id, level='session_id')).sum()
    print(s_id, n_na)
    if n_na > 1:
        plt.figure()
        plt.plot(kinematics['frame_id'].xs(s_id, level='session_id').isna().reset_index(drop=True), alpha=1, color='k')
        plt.plot(kinematics['frame_raw'].xs(s_id, level='session_id').isna().reset_index(drop=True), alpha=.4)
        plt.plot(kinematics['frame_yaw'].xs(s_id, level='session_id').isna().reset_index(drop=True), alpha=.4)
        plt.plot(kinematics['frame_pitch'].xs(s_id, level='session_id').isna().reset_index(drop=True), alpha=.4)
        plt.xlabel("Frame index")
        plt.title(f"Session {s_id} with {n_na} NaNs in raw frame")
plt.show()


In [ ]:
plt.close('all')

# Get unique session IDs
session_ids = kinematics.index.get_level_values('session_id').unique()
n_sessions = len(session_ids)

# Calculate grid dimensions (5 columns)
n_cols = 5
n_rows = int(np.ceil(n_sessions / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3), 
                         squeeze=False, sharex=True, sharey=True)
axes = axes.flatten()

for i, s_id in enumerate(session_ids):
    # if s_id ==5:
    #     continue
    ax = axes[i]
    ax.hist(kinematics['frame_raw_500msMedian'].xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='raw', density=True)
    ax.hist(kinematics['frame_yaw_500msMedian'].abs().xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='yaw', density=True)
    ax.hist(kinematics['frame_pitch_500msMedian'].abs().xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='pitch', density=True)
    
    # ax.hist(kinematics['frame_raw'].xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='raw', density=True)
    # ax.hist(kinematics['frame_velocity'].abs().xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='velocity', density=True)
    # ax.hist(kinematics['frame_yaw'].abs().xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='yaw', density=True)
    # ax.hist(kinematics['frame_pitch'].abs().xs(s_id, level='session_id').values.clip(-5,200), bins=100, alpha=0.3, label='pitch', density=True)
    
    # ax.set_xlim(-5, 250)
    plt.yscale('log')
    ax.legend()
    ax.set_title(f"Session {i} " + str(s_id))
    # break

plt.tight_layout()
plt.show()


In [ ]:
plt.close('all')
cols = ['frame_velocity', 'frame_acc', ]#'frame_raw', 'frame_yaw', 'frame_pitch', 'frame_RawYawPitch_abs_velocity_sum']
for col in cols:
    plt.figure(figsize=(3,3))
    # mask = kinematics[col] >250
    # plt.scatter(kinematics[col][mask].values, kinematics.index.get_level_values('session_id')[mask])
    for s_id in kinematics.index.get_level_values('session_id').unique():
        # if s_id <8:
        #     continue
        plt.hist(kinematics[col].xs(s_id, level='session_id').values, bins=100, alpha=0.3, label=s_id)
    # plt.hist(kinematics[col].values, bins=100)
    plt.yscale('log')
    plt.legend()
    plt.title(col)
    plt.show()
    # break

In [ ]:
kinematics[['frame_position']].max()

In [ ]:
plt.close('all')
dat = kinematics[['track_zone', 'frame_position']]
# sample
# dat = np.random.choice(dat.values, size=1_000_000, replace=False)
print(dat['track_zone'])
for zone in dat['track_zone'].dropna().unique():
    # print(dat['frame_position'][dat['track_zone']])
    positions = dat['frame_position'][dat['track_zone']==zone].dropna()
    plt.hist(positions, bins=100, alpha=0.5, )#label=str(zone))
plt.legend()
plt.yscale('log')
# plt.scatter(dat['frame_position'], dat['frame_id'], 
#             c=dat['track_zone'].cat.codes, label=dat['track_zone'], 
#             cmap='tab20')
# plt.colorbar(label='track_zone')
# framew_beh.columns.tolist()

# lines = {
#         'cueZone_visible': -120,
#         'cueZone_entry': -80,
#         'cueZone_exit': 10,
#         'enter_reward1Zone': 50,
#         'enter_reward2Zone': 170,}

zones = {
    'startZone': (-169,-120),
    'visibleCue': (-120,-80),
    'nextToCue': (-80,10),
    'afterCue': (10, 50),
    'reward1Zone': (50,110),
    'bewteenRewardZones': (110,170),
    'reward2Zone': (170,230),
    'endZone': (230,260),
    'ITI': (260,265),
}

# use span
for z in zones:
    color = np.random.rand(3,)
    plt.axvspan(*zones[z], color=color, alpha=0.2, label=z)

plt.legend()
plt.show()


In [ ]:
plt.close('all')

# Get unique session IDs
session_ids_unique = kinematics.index.get_level_values(2).unique()

for j, session_id in enumerate(session_ids_unique):
    # Select one session
    kin = kinematics.loc[(slice(None), slice(None), session_id),]
    
    # Get unique trial IDs
    trial_ids = kin.trial_id.unique()
    n_trials = len(trial_ids)
    print(f"Session {session_id}: {n_trials} trials")
    
    # Calculate grid dimensions
    n_cols = min(12, n_trials)  # Max 12 columns
    n_rows = int(np.ceil(n_trials / n_cols))
    
    # Create figure with shared axes
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.2, n_rows * 1.2), 
                             sharex=True, sharey=True, squeeze=False)
    axes = axes.flatten()
    
    for i, trial_id in enumerate(trial_ids):
        ax = axes[i]
        
        # Filter data for this trial
        trial_kin = kin[kin.trial_id == trial_id]
        unity_vel = trial_kin.frame_velocity
        ball_vel = trial_kin.frame_raw
        abs_sum_ball_vel = trial_kin.frame_RawYawPitch_abs_vel_sum
        
        # Remove disabled movement frames
        disabled_movement_mask = unity_vel < 0.001
        unity_vel = unity_vel[~disabled_movement_mask]
        ball_vel = ball_vel[~disabled_movement_mask]
        abs_sum_ball_vel = abs_sum_ball_vel[~disabled_movement_mask]
        
        if len(unity_vel) > 0:
            color = trial_kin.frame_id[~disabled_movement_mask]
            ax.scatter(unity_vel, ball_vel, c=color, alpha=0.15, s=3)
            # ax.scatter(unity_vel, abs_sum_ball_vel, c=color, alpha=0.15, s=3)
            ax.plot([0, 100], [0, 100], 'k--', linewidth=0.5)
        
        ax.set_title(f'Trial {trial_id}', fontsize=6, pad=1)
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        
        # Add grid, remove spines
        ax.grid(True, alpha=0.3, linewidth=0.3)
        for spine in ax.spines.values():
            spine.set_visible(False)
        # ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)jachudi
        
        
        
    
    # Hide unused subplots
    for i in range(n_trials, len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(f'Session {j}, {session_id}', fontsize=10, y=0.99)
    plt.subplots_adjust(wspace=0.12, hspace=0.25, left=0.02, right=0.98, top=0.92, bottom=0.02)
    # save them in output dir
    plt.savefig(f"{output_dir}/vel_correlation_session_{session_id}.png", dpi=300)
    # if session_id < 3:
    #     plt.show()
    # else:
    #     plt.close(fig)
    # break
plt.show()

In [ ]:
# Calculate Pearson R for each session-trial combination
from scipy.stats import pearsonr

# Get unique session IDs
session_ids_unique = kinematics.index.get_level_values(2).unique()

# Dictionary to store correlations
correlations = []

for session_id in session_ids_unique:
    # Select one session
    kin = kinematics.loc[(slice(None), slice(None), session_id),]
    
    # Get unique trial IDs
    trial_ids = kin.trial_id.unique()
    
    for trial_id in trial_ids:
        # Filter data for this trial
        trial_kin = kin[kin.trial_id == trial_id]
        unity_vel = trial_kin.frame_velocity.clip(-20, 2000)
        ball_vel = trial_kin.frame_raw
        
        # Remove disabled movement frames
        disabled_movement_mask = (unity_vel < 0.001) | unity_vel.isna()
        unity_vel = unity_vel[~disabled_movement_mask]
        ball_vel = ball_vel[~disabled_movement_mask]
        
        # Calculate Pearson R if we have enough data
        if len(unity_vel) > 1:
            r, p_value = pearsonr(unity_vel, ball_vel / 10.8)
            correlations.append({
                'session_id': session_id,
                'trial_id': trial_id,
                'pearson_r': r,
                'p_value': p_value
            })

# Convert to DataFrame
corr_df = pd.DataFrame(correlations)

# Pivot to create a matrix: sessions as rows, trials as columns
corr_matrix = corr_df.pivot(index='session_id', columns='trial_id', values='pearson_r')

# Create heatmap
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(corr_matrix.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

# Set ticks and labels
ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_yticks(np.arange(len(corr_matrix.index)))
ax.set_xticklabels(corr_matrix.columns, fontsize=6)
ax.set_yticklabels(corr_matrix.index)

# Rotate the tick labels for better readability
plt.setp(ax.get_xticklabels(), rotation=90, ha="center")

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Pearson R', rotation=270, labelpad=15)

# Add title and labels
ax.set_xlabel('Trial ID')
ax.set_ylabel('Session ID')
ax.set_title('Pearson Correlation between Unity Velocity and Ball Velocity\n(per Trial per Session)', 
             pad=20)

# Add grid
ax.set_xticks(np.arange(len(corr_matrix.columns))-.5, minor=True)
ax.set_yticks(np.arange(len(corr_matrix.index))-.5, minor=True)
ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.savefig(f"{output_dir}/pearson_r_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print(f"\nPearson R Summary Statistics:")
print(f"Mean R: {corr_df['pearson_r'].mean():.3f}")
print(f"Median R: {corr_df['pearson_r'].median():.3f}")
print(f"Min R: {corr_df['pearson_r'].min():.3f}")
print(f"Max R: {corr_df['pearson_r'].max():.3f}")
print(f"Std R: {corr_df['pearson_r'].std():.3f}")

## PART 2

In [ ]:
framew_beh = kinematics
framew_beh

In [ ]:
plt.close('all')
fps = 60

def ballvel_plots(plottype):
    # plottype = 'var_hist'
    window = 6  # over ballvel packages
    
    # plottype = 'hist'
    # plottype = 'line'
    # for line args
    arround = 50_000
    delta_bv_samples = 24*60
    from_i, to_i = arround-delta_bv_samples, arround+delta_bv_samples
    
    # Create figure with subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5), sharex=True, sharey=True)
    axes = axes.flatten() if n_sessions > 1 else [axes]
    
    for i, fullsession_name in enumerate(session_dirs):
        kwargs = {'start': from_i, 'stop': to_i} if plottype == 'line' else {}
        
        ball_vels = session_modality_from_nas(fullsession_name, 'ballvelocity', 
                                              columns=['ballvelocity_raw', 'ballvelocity_yaw', 'ballvelocity_pitch', 'ballvelocity_pc_timestamp'],
                                              **kwargs)
        if ball_vels.shape[0] == 0:
            axes[i].axis('off')
            continue
        
        ax = axes[i]
        bv_samples_per_sec = int(1/(np.diff(ball_vels['ballvelocity_pc_timestamp']).mean() / 1e6))
        bv_samples_per_frame = bv_samples_per_sec // fps
        print(f"session {i} Ball samples per frame: {bv_samples_per_frame}")
        
        raw = ball_vels['ballvelocity_raw'].clip(-20, 80)
        yaw = ball_vels['ballvelocity_yaw'].clip(-20, 80)
        pitch = ball_vels['ballvelocity_pitch'].clip(-20, 80)
        ryp = (raw+yaw+pitch).abs()
        ryp_framewise = ryp.rolling(window=bv_samples_per_frame).mean()
        
        if plottype == 'line':
            ax.plot(raw, alpha=0.5, label='forward', color='blue')
            ax.plot(yaw, alpha=0.5, label='sidways', color='purple')
            ax.plot(pitch, alpha=0.5, label='rotation', color='teal')
            ax.plot(ryp_framewise, alpha=1, label='abs-sum-smoothed', color='red', linestyle='--')
            ax.vlines(x=ryp.index[::fps], ymin=-10, ymax=10, color='black', linestyle='--', alpha=0.5)
            ax.set_xticks(ryp.index[::fps*bv_samples_per_frame], labels=range(len(ryp.index[::fps*bv_samples_per_frame])))
            ax.set_xlabel('Time (s)', fontsize=9)
        
        elif plottype == 'hist':
            ax.hist(raw, alpha=0.5, bins=60, label='forward', 
                    density=True, color='blue')
            ax.hist(yaw, alpha=0.5, bins=60, label='sidways', 
                    density=True, color='purple')
            ax.hist(pitch, alpha=0.5, bins=60, label='rotation', 
                    density=True, color='teal')
            ax.hist(ryp_framewise, alpha=0.8, bins=60, label='abs-sum-smoothed', histtype='step', 
                    density=True, color='red')
            ax.set_ylim(0, .3)
            ax.set_xlabel('Velocity', fontsize=9)
        
        elif plottype == 'var_hist':
            raw_var = raw.abs().rolling(window=window).std().clip(0, 6)
            yaw_var = yaw.abs().rolling(window=window).std().clip(0, 6)
            pitch_var = pitch.abs().rolling(window=window).std().clip(0, 6)
            ax.hist(raw_var, alpha=0.8, bins=60, label='forward', 
                    density=True, color='blue', histtype='step')
            ax.hist(yaw_var, alpha=0.8, bins=60, label='sidways', 
                    density=True, color='purple', histtype='step')
            ax.hist(pitch_var, alpha=0.8, bins=60, label='rotation', 
                    density=True, color='teal', histtype='step')
            ax.hist((raw_var+yaw_var+pitch_var)/3, alpha=0.3, bins=60, 
                    label='sum', density=True, color='red')
            ax.set_xlabel('Variability', fontsize=9)
        
        ax.set_ylabel('Density', fontsize=9)
        ax.set_title(f'Session {i}', fontsize=10, pad=8)
        ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    fig.suptitle(f'Ball Velocity - {plottype} (window={window})', fontsize=14, y=0.995)
    plt.tight_layout()
    plt.show()

# Call the function for each plot type
for plottype in ['line', 'hist', 'var_hist']:
    ballvel_plots(plottype=plottype)

In [ ]:
plt.close('all')

# Get all unique sessions
session_ids = framew_beh.index.get_level_values('session_id').unique()
n_sessions = len(session_ids)

# Set up grid: 5 columns
n_cols = 5
n_rows = int(np.ceil(n_sessions / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5), sharex=True, sharey=True)
axes = axes.flatten() if n_sessions > 1 else [axes]

window = 8

for i, session_id in enumerate(session_ids):
    ax = axes[i]
    
    # Select the data for the session
    session_data = framew_beh.loc[(slice(None), slice(None), session_id)]
    forward_vel = session_data['frame_raw']
    rotation_vel = session_data['frame_yaw'].abs() + session_data['frame_pitch'].abs()
    summed_abs_vel = session_data['frame_RawYawPitch_abs_vel_sum']
    
    corr, var_fwd, var_rot = calculate_rolling_metrics(forward_vel, rotation_vel, window)
    
    # Plot histogram for this session
    ax.hist(corr.clip(-1, 10).dropna(), alpha=0.5, bins=60, label='Correlation', 
            density=True, color='green')
    ax.hist(var_fwd.clip(-1, 10).dropna(), alpha=0.5, bins=60, label='Var Forward', 
            density=True, color='blue')
    ax.hist(var_rot.clip(-1, 10).dropna(), alpha=0.5, bins=60, label='Var Rotation', 
            density=True, color='purple')
    
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title(f'Session {i}: {session_id}', fontsize=10, pad=8)
    ax.legend(loc='upper right', fontsize=7, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_ylim(0,3)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Rolling Metrics across Sessions (window={window})', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')

# Get all unique sessions
session_ids = framew_beh.index.get_level_values('session_id').unique()
n_sessions = len(session_ids)

# Set up grid: 5 columns
n_cols = 5
n_rows = int(np.ceil(n_sessions / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5), sharex=True, sharey=True)
axes = axes.flatten() if n_sessions > 1 else [axes]

for i, session_id in enumerate(session_ids):
    ax = axes[i]
    
    # Select the data for the session
    session_data = framew_beh.loc[(slice(None), slice(None), session_id)]
    session_data = session_data[session_data['track_zone'] != 'reward1Zone']
    forward_vel = session_data['frame_raw'].abs()
    rotation_vel = session_data['frame_yaw'].abs() + session_data['frame_pitch'].abs()
    summed_abs_vel = session_data['frame_RawYawPitch_abs_vel_sum']
    
    # Calculate proportions
    low_vel_mask = (summed_abs_vel < 35) & (summed_abs_vel > 3)
    rot_proportion = (rotation_vel.dropna() / (summed_abs_vel.dropna() + 0.1)).clip(0, 1)
    frwd_proportion = (forward_vel.dropna() / (summed_abs_vel.dropna() + 0.1)).clip(0, 1)
    
    bins = 40
    bin_range = (0, 1)
    
    # Low velocity - filled histograms
    ax.hist(rot_proportion[low_vel_mask], alpha=0.5, bins=bins, range=bin_range,
            label='Rot (low vel)', color='red', edgecolor='none', density=True)
    ax.hist(frwd_proportion[low_vel_mask], alpha=0.5, bins=bins, range=bin_range,
            label='Fwd (low vel)', color='green', edgecolor='none', density=True)
    
    # # High velocity - step histograms
    # ax.hist(rot_proportion[high_vel_mask], alpha=1, bins=bins, range=bin_range,
    #         label='Rot (high vel)', histtype='step', linewidth=2, color='darkred')
    # ax.hist(frwd_proportion[high_vel_mask], alpha=1, bins=bins, range=bin_range,
    #         label='Fwd (high vel)', histtype='step', linewidth=2, color='darkgreen')
    
    ax.set_xlabel('Proportion of Total Velocity', fontsize=9)
    ax.set_ylabel('Count', fontsize=9)
    ax.set_title(f'S{i}: {session_id}', fontsize=10, pad=8)
    ax.legend(loc='upper right', fontsize=6, framealpha=0.9, ncol=2)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(0, 1)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Forward vs Rotation Components at R1 Zone (velocity threshold: 35 cm/s)', 
             fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
plt.close('all')

# Get all unique sessions
session_ids = framew_beh.index.get_level_values('session_id').unique()
n_sessions = len(session_ids)

# Set up grid: 5 columns
n_cols = 5
n_rows = int(np.ceil(n_sessions / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5), sharex=True, sharey=True)
axes = axes.flatten() if n_sessions > 1 else [axes]

# Define velocity bin edges
edges = np.array([5, 17, 29, 50, 75, 150])
bar_width = 0.8  # Constant width for all bars
use_density = True  # Set to True for normalized density (area sums to 1), False for raw counts

for i, session_id in enumerate(session_ids):
    ax = axes[i]
    
    # Select the data for the session
    session_data = framew_beh.loc[(slice(None), slice(None), session_id)]
    session_data = session_data[session_data['track_zone'] == 'reward1Zone']
    reward_thr = (session_data['velocity_threshold_at_R1'] * session_data['fps']).dropna().astype(int).mode().item()
    print(reward_thr)
    
    forward_vel = session_data['frame_raw'].abs()
    yaw_vel = session_data['frame_yaw'].abs()
    pitch_vel = session_data['frame_pitch'].abs()
    summed_abs_vel = session_data['frame_RawYawPitch_abs_vel_sum']
    
    # Remove NaN values
    valid_mask = ~(forward_vel.isna() | yaw_vel.isna() | pitch_vel.isna() | summed_abs_vel.isna())
    forward_vel = forward_vel[valid_mask]
    yaw_vel = yaw_vel[valid_mask]
    pitch_vel = pitch_vel[valid_mask]
    summed_abs_vel = summed_abs_vel[valid_mask]
    
    # Bin the velocities
    bin_indices = np.digitize(summed_abs_vel, edges)
    
    # Calculate total samples for density normalization
    total_samples = len(summed_abs_vel)
    
    # Prepare data for stacked bar plot
    bar_positions = []
    forward_heights = []
    yaw_heights = []
    pitch_heights = []
    bin_labels = []
    
    for bin_idx in range(1, len(edges)):
        mask = bin_indices == bin_idx
        count = mask.sum()
        
        if count == 0:
            continue
        
        # Get velocities in this bin
        fwd_in_bin = forward_vel[mask]
        yaw_in_bin = yaw_vel[mask]
        pitch_in_bin = pitch_vel[mask]
        
        # Calculate total contributions
        total_fwd = fwd_in_bin.sum()
        total_yaw = yaw_in_bin.sum()
        total_pitch = pitch_in_bin.sum()
        total = total_fwd + total_yaw + total_pitch
        
        if total > 0:
            # Bar position at bin index
            bar_positions.append(bin_idx)
            
            if use_density:
                # Proper histogram density: area = height * width = count / total_samples
                # So height = count / (total_samples * bin_width)
                bin_width_actual = edges[bin_idx] - edges[bin_idx - 1]
                height_base = count / (total_samples)
            else:
                # Raw count
                height_base = count
            
            # Heights: split by component proportions
            forward_heights.append(height_base * (total_fwd / total))
            yaw_heights.append(height_base * (total_yaw / total))
            pitch_heights.append(height_base * (total_pitch / total))
            bin_labels.append(f'{edges[bin_idx-1]}-{edges[bin_idx]}')
    
    # Create stacked bar plot
    if len(bar_positions) > 0:
        ax.bar(bar_positions, forward_heights, width=bar_width, 
               alpha=0.5, color='green', edgecolor='black', linewidth=0.5, label='Forward')
        ax.bar(bar_positions, yaw_heights, width=bar_width, 
               bottom=forward_heights, alpha=0.5, color='salmon', 
               edgecolor='black', linewidth=0.5, label='sideways')
        # Calculate bottom for pitch (forward + yaw)
        pitch_bottom = [forward_heights[j] + yaw_heights[j] for j in range(len(bar_positions))]
        ax.bar(bar_positions, pitch_heights, width=bar_width, 
               bottom=pitch_bottom, alpha=0.5, color='lightcoral', 
               edgecolor='black', linewidth=0.5, label='rotation')
    
    # Add vertical line at reward threshold (interpolated position within bin)
    threshold_bin = np.digitize(reward_thr, edges)
    if threshold_bin > 0 and threshold_bin < len(edges):
        # Map threshold value to x-axis position (interpolate within the bin range)
        fraction = (reward_thr - edges[threshold_bin-1]) / (edges[threshold_bin] - edges[threshold_bin-1])
        x_pos = (threshold_bin - 0.5) + fraction
        ax.axvline(x=x_pos, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'R1 thr ({reward_thr})')
    
    ax.set_xlabel('Velocity Bin', fontsize=9)
    ax.set_ylabel('Density' if use_density else 'Count', fontsize=9)
    ax.set_title(f'S{i}: {session_id}', fontsize=10, pad=8)
    ax.legend(loc='upper right', fontsize=6, framealpha=0.9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks(range(1, len(edges)))
    ax.set_xticklabels([f'{edges[j]}-{edges[j+1]}' for j in range(len(edges)-1)], 
                        rotation=45, ha='right', fontsize=7)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

title_mode = 'Normalized Histogram (area sums to 1)' if use_density else 'Bar Height = Count'
fig.suptitle(f'Forward vs Rotation Components by Velocity Bins ({title_mode})', 
             fontsize=14, y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.mixture import GaussianMixture

xmin,xmax = 2.5, 200
std = 2

def plot_gmm_prop_fwrd(session_data, ax, title=''):
    
    main_metric = 'frame_RawYawPitch_abs_vel_sum'
    total_sum = session_data[main_metric].copy()#.clip(0, 200)
    large_vel_mask = total_sum >xmax
    # sample jitter in x 
    x_jitter = np.random.randn(large_vel_mask.sum()) * std
    
    total_sum.loc[large_vel_mask] = (total_sum[large_vel_mask].clip(xmin,xmax) + x_jitter).values
    total_sum.loc[total_sum<xmin] = np.nan
            
    # what prop of the sum is forward driven, what prop is sideway or rotation driven
    frwd_prop = (session_data['frame_raw'].abs() / total_sum).replace([np.inf, -np.inf], np.nan).fillna(0)
    # sideway_prop = (trial_data['frame_yaw'].abs() / total_sum).replace([np.inf, -np.inf], np.nan).fillna(0)
    # rotation_prop = (trial_data['frame_pitch'].abs() / total_sum).replace([np.inf, -np.inf], np.nan).fillna(0)
    
    zone_mask = session_data['track_zone'].isin(['reward1Zone', 'reward2Zone']).values
    x = total_sum[zone_mask]
    y = frwd_prop[zone_mask].clip(0,1).values
    
    # correct zone
    
    ax.scatter(x, y, color='k', s=2, label='forward prop', alpha=0.1)
    
    # labels
    avg_prop_rew_vel = y[((x>2.5) & (x<20))].mean()
    avg_prop_low_vel = y[((x>20) & (x<45))].mean()
    
    ax.plot()
    
    ax.vlines([3,20], 0,1, color='green', label=f'Avg. forward prop: {avg_prop_rew_vel:.2f}')
    ax.vlines([21,45], 0,1, color='green', alpha=.4, label=f'Avg. forward prop: {avg_prop_low_vel:.2f}')
    ax.plot([3,20], [avg_prop_rew_vel,avg_prop_rew_vel], color='green', linestyle='dashed', linewidth=3)
    ax.plot([21,45], [avg_prop_low_vel,avg_prop_low_vel], color='green', alpha=.4, linestyle='dashed', linewidth=3)
        # ---- text annotations ----
    ax.text(3, avg_prop_low_vel + 0.03,
            f'mean: {avg_prop_low_vel:.2f}',
            color='k', fontsize=9, ha='left', va='bottom')

    ax.text(22, avg_prop_rew_vel + 0.03,
            f'mean:{avg_prop_rew_vel:.2f}',
            color='k', fontsize=10, ha='left', va='bottom')

    # Fit a 2D GMM (k=4) to (main_metric, forward_prop) and draw density overlay
    try:
        X = np.column_stack([x, y])
        # remove any rows with NaN or inf
        mask = np.isfinite(X).all(axis=1)
        Xf = X[mask]
        if Xf.shape[0] >= 4:
            gmm = GaussianMixture(n_components=10, covariance_type='full', random_state=42)
            gmm.fit(Xf)
            # create grid for density evaluation
            x_min, x_max = Xf[:,0].min(), Xf[:,0].max()
            y_min, y_max = max(0, Xf[:,1].min()), min(1, Xf[:,1].max())
            x_range = np.linspace(x_min, x_max, 200)
            y_range = np.linspace(y_min, y_max, 200)
            xx, yy = np.meshgrid(x_range, y_range)
            grid = np.column_stack([xx.ravel(), yy.ravel()])
            logprob = gmm.score_samples(grid)
            density = np.exp(logprob).reshape(xx.shape)
            # normalize density for plotting
            density = density / (density.max() + 1e-12)
            # overlay density as a shaded contourf
            ax.contourf(xx, yy, density, levels=8, cmap='Reds', alpha=0.45)
            # optional: contour lines
            ax.contour(xx, yy, density, levels=6, colors='k', linewidths=0.5, alpha=0.6)
            # colorbar for density (create a mappable)
            # Note: contourf returns QuadContourSet; use it for colorbar if desired
            # plt.xlim(xmin,xmax+5)
            ax.legend()
    except Exception as e:
        print('GMM overlay failed:', e)
    ax.set_title(title)

plt.close('all')

# Get all unique sessions
session_ids = framew_beh.index.get_level_values('session_id').unique()
n_sessions = len(session_ids)

# Set up grid: 5 columns
n_cols = 5
n_rows = int(np.ceil(n_sessions / n_cols))

# Create figure with subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5), sharex=True, sharey=True)
axes = axes.flatten() if n_sessions > 1 else [axes]

window = 8

for i, session_id in enumerate(session_ids):
    ax = axes[i]
    print(f"Processing session {session_id} ({i+1}/{n_sessions})")    
    
    # Select the data for the session
    session_data = framew_beh.loc[(slice(None), slice(None), session_id)]
    title = f"S{i:02d}"
    plot_gmm_prop_fwrd(session_data, ax, title)
    # break

# # Hide unused subplots
# for j in range(i + 1, len(axes)):
#     axes[j].axis('off')

fig.suptitle(f'Rolling Metrics across Sessions (window={window})', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()